# Direct String Matching and Result Contracts


```{contents}
:local:
:depth: 2
```


```{index} Define What Counts as a Match Before Choosing an Algorithm
```



```{index} Separate a Candidate Start from Its Matched Length
```

## Separate a Candidate Start from Its Matched Length


A direct matcher asks the same small question at each legal start: do all pattern code units agree here? There are two positions to track. Start chooses a candidate in the text; offset chooses a position inside the pattern. A mismatch rejects the current candidate only. The next start may still match, including an occurrence overlapping the previous match.

For text ABABA and pattern ABA:

| Candidate start | Character comparisons | Decision |
| --- | --- | --- |
| 0 | A=A, B=B, A=A | Report 0 |
| 1 | B differs from A | Reject only this start |
| 2 | A=A, B=B, A=A | Report 2 |

The final legal nonempty start is text.Length-pattern.Length. A longer pattern has no legal start. The empty-pattern policy is separate: this book reports every boundary, including the one after the final text code unit. The focused example below assumes non-null strings; the retained full API also rejects null inputs explicitly.


In [1]:
using System;
using System.Collections.Generic;
static int[] Direct(string text, string pattern)
{
    var starts = new List<int>();
    for (int start = 0; start <= text.Length - pattern.Length; start++)
    {
        int offset = 0;
        while (offset < pattern.Length && text[start + offset] == pattern[offset]) offset++;
        if (offset == pattern.Length) starts.Add(start);
    }
    return starts.ToArray();
}
Console.WriteLine($"Overlaps: {string.Join(",", Direct("ABABA", "ABA"))}");
Console.WriteLine($"Longer count: {Direct("AB", "ABC").Length}");
Console.WriteLine($"Empty starts: {string.Join(",", Direct("AB", ""))}");


Overlaps: 0,2
Longer count: 0
Empty starts: 0,1,2


Offset equals pattern.Length only when the whole pattern has matched. On a mismatch it stops earlier, so the candidate is not reported. Increasing start by one preserves overlaps. For the empty pattern, the inner loop runs zero times, offset already equals zero, and starts zero through text.Length are reported under the stated boundary policy.

Guided task: trace AA in AAAA and fill the starts without running code. Then change the outer update to advance by pattern.Length after a match. Which required starts disappear, and why is the modification incorrect for this contract?

```{dropdown} Check the overlap policy
The required starts are zero, one and two. Skipping two positions after a match loses start one. A non-overlapping API could choose that policy, but it would be solving a different problem. Tests must compare equivalent contracts.
```

For independent comparison with KMP, check empty text, empty pattern, longer patterns, repeated symbols, absent patterns and case differences before counting work. These examples compare ordinal UTF-16 code units. They do not normalize text or implement a culture-aware search. The retained hop-versus-cost discussion below is an objective-contract comparison linked to the graph chapters, not a prerequisite for the string algorithm.


```{index} Define What Counts as a Match Before Choosing an Algorithm
```

## Define What Counts as a Match Before Choosing an Algorithm



A message filter searches text for a literal pattern. Decide whether matching is case-sensitive, whether all overlapping starts are required, and how an empty pattern is treated. These choices determine the correct result before any comparison-count claim can be made. The core examples compare ordinal UTF-16 code units; a returned position is an offset in that representation, not necessarily a user-perceived character count.

For text ABABA and pattern ABA, the starts are zero and two. Moving the candidate start by the pattern length after the first match would miss the overlapping occurrence at two. A direct matcher instead considers every candidate start whose remaining text is long enough, comparing pattern elements until a mismatch or a complete match.

| Candidate start | Comparison outcome | Report it? |
| --- | --- | --- |
| 0 | ABA matches ABA | Yes |
| 1 | B differs from A immediately | No |
| 2 | ABA matches ABA | Yes |

The direct algorithm's simplicity makes it a useful independent reference for more elaborate matching. Its contract must still handle an empty pattern. Under the all-boundaries convention used here, an empty pattern matches before the first code unit, between adjacent code units, and after the final code unit. Thus text of length n has n+1 empty-pattern starts. Another API can choose another contract, but comparisons must use the same one.

For related explanation, see the further reading below.[^development-reading]


```{index} Use Mismatches as Evidence
```



```{index} Use Mismatches as Evidence
```

## Use Mismatches as Evidence



A mismatch at one candidate start establishes only that this candidate fails. It does not prove that the text contains no later occurrence. The outer loop advances the start; the inner loop advances the pattern position. Confusing those two positions can skip starts or compare the wrong slices.

With text length n and pattern length m, a direct comparison model can perform up to proportional to n*m work for nonempty patterns within the text. A long shared prefix followed by a mismatch can repeat work at many starts. The next lesson develops how prefix information can avoid some repetition. For short patterns or a small number of queries, a clear direct approach or a suitable library operation can still be a reasonable engineering choice; a more sophisticated algorithm is not required merely because it exists.

Guided check: find all starts of AA in AAAA, then an absent pattern longer than the text, then the empty pattern. Derive expected outputs manually. Use those cases when checking the preserved implementations so that a fast candidate cannot quietly change overlap or boundary semantics. Keep weighted-graph hop/cost reminders as links to Chapters 20 and 24; they are separate result-contract examples, not prerequisites for reading a string matcher.


```{index} Direct Matching with an Explicit Contract
```

## Direct Matching with an Explicit Contract




```{index} Direct Matching with an Explicit Contract
```

Search exact ordinal UTF-16 code units, returning every start position, including overlapping matches. A missing nonempty pattern returns an empty array. An empty pattern matches each boundary 0..text.Length; null arguments are rejected. Derived first-match clients return the first position or −1. Counts include every executed code-unit equality test, including mismatches.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

foreach(var pair in new[]{("aaaaa","aaa"),("abracadabra","cad"),("abc",""),("","x")})
{
 var result=PatternScan.Direct(pair.Item1,pair.Item2);
 Console.WriteLine($"text=[{pair.Item1}], pattern=[{pair.Item2}], positions=[{string.Join(",",result.Positions)}], comparisons={result.Comparisons}");
}
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


text=[aaaaa], pattern=[aaa], positions=[0,1,2], comparisons=9
text=[abracadabra], pattern=[cad], positions=[4], comparisons=11
text=[abc], pattern=[], positions=[0,1,2,3], comparisons=0
text=[], pattern=[x], positions=[], comparisons=0


The loop tests each legal start once. At offset j, the first j code units match; a mismatch rules out that start, while j=m proves an occurrence. For n text units and m>0 pattern units, worst-case comparison work is O((n−m+1)m) when m≤n, commonly written O(nm); m>n gives zero tests. Returned r positions cost O(r) storage. Empty patterns require O(n) output despite zero equality tests.

**Common mistake.** Advancing the start position by the pattern length after a match. That skips overlapping occurrences, so `aaa` in `aaaaa` reports one position instead of three; the contract says every legal start is tested, so the start advances by one.


```{index} Count the Work of Direct Matching
```

### Count the Work of Direct Matching

The bound O(nm) describes the worst case, and it helps to know what input produces it and how far ordinary input is from it. A candidate start costs one comparison when the first code unit differs and up to m comparisons when the text agrees with the pattern for a long stretch before failing. The worst case is therefore text and pattern that agree almost everywhere: a text of n copies of one letter searched for a pattern of m - 1 copies followed by a different letter does m comparisons at nearly every start, about nm in total. Text in a natural language is nearly the opposite: most starts fail on the first or second comparison, so the direct scan costs little more than n and often beats cleverer algorithms once their preparation is counted. The example counts comparisons for the same pattern length on three kinds of text.

In [ ]:
using System;

int n = 300, m = 10;
string repetitive = new string('a', n);                                  // every start matches m - 1 units then fails
string pattern = new string('a', m - 1) + "b";
var random = new Random(3);
char[] mixed = new char[n]; for (int i = 0; i < n; i++) mixed[i] = (char)('a' + random.Next(4));
string varied = new string(mixed);
string prose = string.Concat(System.Linq.Enumerable.Repeat("the quarterly report lists every invoice and payment ", 6)).Substring(0, n);

Report("repetitive text", repetitive, pattern);
Report("four-letter random", varied, "aaaaaaaaab");
Report("English-like prose", prose, "payment an");

void Report(string label, string text, string pat)
{
    long comparisons = 0; int matches = 0;
    for (int start = 0; start + pat.Length <= text.Length; start++)
    {
        int offset = 0;
        while (offset < pat.Length) { comparisons++; if (text[start + offset] != pat[offset]) break; offset++; }
        if (offset == pat.Length) matches++;
    }
    Console.WriteLine($"{label,-19} n={text.Length} m={pat.Length}: comparisons={comparisons,5} ({(double)comparisons / text.Length:F2} per text unit), matches={matches}, n*m={text.Length * pat.Length}");
}

repetitive text     n=300 m=10: comparisons= 2910 (9.70 per text unit), matches=0, n*m=3000
four-letter random  n=300 m=10: comparisons=  378 (1.26 per text unit), matches=0, n*m=3000
English-like prose  n=300 m=10: comparisons=  337 (1.12 per text unit), matches=0, n*m=3000


On repetitive text the scan makes almost m comparisons per text unit, within a few percent of the n × m bound, and finds nothing. On random text over four letters it averages a little over one comparison per unit, because a start usually fails at once and only one in four survives to the second comparison. On the prose it is close to one per unit as well. The worst case is real, repeated symbols occur in log files, DNA, and padded records, but it is specific, and the prefix-reuse algorithm of the next section is justified by that case rather than by typical text, where both algorithms cost about n and the direct scan has no preparation to pay for.

```{index} Code Tracing: Comparisons per Start
```

### Code Tracing: Comparisons per Start

Trace the direct matcher for pattern `AAB` in text `AAAABAAB`. For each candidate start write the number of comparisons made and whether the start is reported. Predict the total number of comparisons and the reported starts, then run the solution. Which starts cost the full three comparisons without being reported?

In [ ]:
// Solution
using System;
using System.Collections.Generic;

string text = "AAAABAAB", pattern = "AAB";
long total = 0;
var reported = new List<int>();
for (int start = 0; start + pattern.Length <= text.Length; start++)
{
    int offset = 0, comparisons = 0;
    while (offset < pattern.Length) { comparisons++; if (text[start + offset] != pattern[offset]) break; offset++; }
    bool match = offset == pattern.Length;
    if (match) reported.Add(start);
    total += comparisons;
    Console.WriteLine($"start {start}: {comparisons} comparison(s), {(match ? "report" : "reject")}");
}
Console.WriteLine($"total comparisons={total}, reported starts={string.Join(",", reported)}");

start 0: 3 comparison(s), reject
start 1: 3 comparison(s), reject
start 2: 3 comparison(s), report
start 3: 2 comparison(s), reject
start 4: 1 comparison(s), reject
start 5: 3 comparison(s), report
total comparisons=15, reported starts=2,5


Starts 2 and 5 are reported after three comparisons each. Starts 0 and 1 also cost three comparisons: the text agrees with `AA` and fails only on the third unit, which is the pattern of wasted work the next section's algorithm removes, because after reading `AAA` the matcher already knows that the next candidate begins with `AA`. Start 3 fails on its second comparison and start 4 on its first. Fifteen comparisons for six starts is already two and a half per start, and a longer run of A's would push it toward three.

```{index} Code Completion: The Inner Comparison Loop
```

### Code Completion: The Inner Comparison Loop

The outer loop over candidate starts is written; the inner comparison is missing. Complete it so that `offset` ends equal to `pattern.Length` exactly when the whole pattern matches at `start`. The expected output is `0,2` for `ABA` in `ABABA`, `0,1,2` for `AA` in `AAAA`, and an empty list for a pattern longer than the text.

In [ ]:
using System;
using System.Collections.Generic;

Console.WriteLine(string.Join(",", Direct("ABABA", "ABA")));
Console.WriteLine(string.Join(",", Direct("AAAA", "AA")));
Console.WriteLine(string.Join(",", Direct("AB", "ABC")));

static List<int> Direct(string text, string pattern)
{
    var starts = new List<int>();
    for (int start = 0; start + pattern.Length <= text.Length; start++)
    {
        int offset = 0;
        // Your code starts here.

        // Your code ends here.
        if (offset == pattern.Length) starts.Add(start);
    }
    return starts;
}

In [ ]:
// Solution
using System;
using System.Collections.Generic;

Console.WriteLine(string.Join(",", Direct("ABABA", "ABA")));
Console.WriteLine(string.Join(",", Direct("AAAA", "AA")));
Console.WriteLine(string.Join(",", Direct("AB", "ABC")));

static List<int> Direct(string text, string pattern)
{
    var starts = new List<int>();
    for (int start = 0; start + pattern.Length <= text.Length; start++)
    {
        int offset = 0;
        while (offset < pattern.Length && text[start + offset] == pattern[offset]) offset++;   // stop at the first disagreement
        if (offset == pattern.Length) starts.Add(start);
    }
    return starts;
}

0,2
0,1,2


The condition tests the bound before the characters, so `text[start + offset]` is read only while `offset` is inside the pattern; the outer loop's `start + pattern.Length <= text.Length` guarantees the text index is also in range. Leaving the block empty keeps `offset` at 0, so a nonempty pattern is never reported and the output is three empty lines. A loop that advanced `start` inside the inner loop would mix the two positions the section separates.

The distinction between minimum hops and minimum cost is developed with its full graph counterexample in {doc}`Shortest Paths and Dijkstra <../24_greedy_graph_optimization/2403_shortest_paths>`. String matching likewise begins by choosing an explicit result contract.


```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. Why must a matcher that reports overlapping occurrences advance the start by one rather than by the pattern length?

```{dropdown} Answer
Because a new occurrence can begin inside the one just found: `ABA` occurs at 0 and 2 in `ABABA`. Skipping the pattern length loses the second. A non-overlapping contract may skip, but it answers a different question and its tests must say so.
```

2. Direct matching is O(nm) in the worst case. On which inputs does it approach that bound, and on which is it close to n?

```{dropdown} Answer
It approaches nm when the text and pattern share long runs of the same units, so most starts fail late; it stays near n on text whose starts usually fail at the first comparison, which is typical of prose and of random text over a reasonable alphabet.
```

3. The empty pattern returns `0, 1, 2, 3` for a three-unit text. Why four positions rather than three?

```{dropdown} Answer
Under the all-boundaries contract the empty pattern matches before the first unit, between each pair, and after the last: n + 1 boundaries. Another contract could return nothing or only 0; the point is that the policy is stated and tested, since the inner loop runs zero times and reports every start otherwise.
```

4. A matcher built with `string.IndexOf(pattern)` in a loop reports different positions from the direct matcher on `aaaa`, `aa`. What is the likely cause?

```{dropdown} Answer
The loop restarts after each match at the position past the match, or uses a culture-sensitive comparison, rather than advancing by one with an ordinal comparison. Both produce a different contract; the direct matcher's positions are the ordinal overlapping ones.
```

```{rubric} Footnotes
```


[^development-reading]: Further reading: [Sedgewick and Wayne, Strings](https://algs4.cs.princeton.edu/50strings/). The worked business scenarios and guided reasoning here are original teaching examples.
